# Full-Text Search in a Database

In production you rarely build an inverted index from scratch. If your documents
already live in a relational database, the simplest option is often to search
them there. This demo builds an inverted index as an ordinary SQL table to make
the structure concrete, then switches to SQLite's built-in FTS5 full-text engine
(a real inverted index with BM25 ranking), and finally shows the equivalent
PostgreSQL syntax that most applications reach for.

**Learning goals:**
- Build an inverted index as a plain SQL table of (term, doc_id, tf) rows
- Answer Boolean and ranked queries with ordinary SQL
- Use SQLite FTS5: a real inverted index with `MATCH` queries and `bm25()` ranking
- Read the PostgreSQL equivalent: `tsvector`, `tsquery`, GIN, and `ts_rank`
- Understand why in-database ranking is good for matching but only adequate for relevance

**Prerequisites:** ch04-01 (inverted index structure), ch04-02 (BM25 scoring)

In [1]:
import sqlite3
from collections import Counter
from shared.library_collection import LIBRARY
from shared.text import pipeline
from shared.display import print_table, display_md

## The Inverted Index as a SQL Table

The chapter makes a key observation: an inverted index is just a set of
(term, document, frequency) rows. A relational database already knows how to
store rows and index a column, so we can build the whole structure as one table
with a B-tree index on the term column. A lookup for a term becomes a `WHERE
term = ?` query, which the index answers without scanning every row.

In [2]:
# Build a fresh in-memory database
con = sqlite3.connect(":memory:")
con.execute("CREATE TABLE postings (term TEXT, doc_id TEXT, tf INTEGER)")
con.execute("CREATE INDEX idx_term ON postings(term)")

# Populate: one row per (term, document) with its term frequency
for key, record in LIBRARY.items():
    tokens = pipeline(f"{record['title']} {record['author']}")
    for term, tf in Counter(tokens).items():
        con.execute("INSERT INTO postings VALUES (?, ?, ?)", (term, key, tf))
con.commit()

n_rows = con.execute("SELECT COUNT(*) FROM postings").fetchone()[0]
n_terms = con.execute("SELECT COUNT(DISTINCT term) FROM postings").fetchone()[0]
display_md(
    f"**Inverted index table built**\n\n"
    f"| Statistic | Value |\n"
    f"|-----------|-------|\n"
    f"| Documents | {len(LIBRARY)} |\n"
    f"| Distinct terms | {n_terms} |\n"
    f"| Postings rows | {n_rows} |"
)

**Inverted index table built**

| Statistic | Value |
|-----------|-------|
| Documents | 50 |
| Distinct terms | 231 |
| Postings rows | 265 |

In [3]:
# The postings list for one term is a single indexed lookup
rows = con.execute(
    "SELECT term, doc_id, tf FROM postings WHERE term = 'comput' ORDER BY doc_id"
).fetchall()
display_md("**Postings for the stemmed term** `comput` (one indexed lookup):")
print_table([list(r) for r in rows], headers=["Term", "Doc ID", "TF"])

**Postings for the stemmed term** `comput` (one indexed lookup):

| Term   | Doc ID                |   TF |
|:-------|:----------------------|-----:|
| comput | computer-networks     |    1 |
| comput | computer-organization |    1 |
| comput | computer-systems      |    1 |
| comput | sicp                  |    1 |
| comput | theory-of-computation |    1 |

## Boolean Queries in SQL

A multi-term AND query is a self-join, or more simply a `GROUP BY` with a
`HAVING` clause that keeps only documents matching every query term.

In [4]:
def boolean_and_sql(terms):
    """Documents containing ALL of the given (already stemmed) terms."""
    placeholders = ",".join("?" for _ in terms)
    sql = (
        f"SELECT doc_id FROM postings WHERE term IN ({placeholders}) "
        f"GROUP BY doc_id HAVING COUNT(DISTINCT term) = ? ORDER BY doc_id"
    )
    return [r[0] for r in con.execute(sql, (*terms, len(terms))).fetchall()]

q_terms = pipeline("computer design")
and_hits = boolean_and_sql(q_terms)
display_md(
    f"**Query:** `computer AND design` -> stemmed `{q_terms}`\n\n"
    f"- Documents matching all terms: {and_hits}"
)

**Query:** `computer AND design` -> stemmed `['comput', 'design']`

- Documents matching all terms: ['computer-organization']

## Ranked Queries in SQL

Boolean matching returns a set. To rank, we join the postings for the query
terms and sum a simple weight per document. Here we sum term frequency; a real
engine would sum a BM25 contribution instead, which the next section shows.

In [5]:
def ranked_tf_sql(terms, limit=10):
    """Rank documents by summed term frequency across the query terms."""
    placeholders = ",".join("?" for _ in terms)
    sql = (
        f"SELECT doc_id, SUM(tf) AS score FROM postings "
        f"WHERE term IN ({placeholders}) GROUP BY doc_id "
        f"ORDER BY score DESC LIMIT ?"
    )
    return con.execute(sql, (*terms, limit)).fetchall()

q_terms = pipeline("computer systems design")
rows = [[i, doc_id, score, LIBRARY[doc_id]["title"][:44]]
        for i, (doc_id, score) in enumerate(ranked_tf_sql(q_terms), 1)]
display_md(f"**Ranked by summed TF for** `computer systems design` -> `{q_terms}`")
print_table(rows, headers=["Rank", "Doc ID", "Sum TF", "Title"])

**Ranked by summed TF for** `computer systems design` -> `['comput', 'system', 'design']`

|   Rank | Doc ID                |   Sum TF | Title                                        |
|-------:|:----------------------|---------:|:---------------------------------------------|
|      1 | computer-systems      |        2 | Computer Systems: A Programmer's Perspective |
|      2 | computer-organization |        2 | Computer Organization and Design: The Hardwa |
|      3 | theory-of-computation |        1 | Introduction to the Theory of Computation    |
|      4 | sicp                  |        1 | Structure and Interpretation of Computer Pro |
|      5 | operating-systems     |        1 | Operating System Concepts                    |
|      6 | everyday-things       |        1 | The Design of Everyday Things                |
|      7 | database-systems      |        1 | Database Systems: The Complete Book          |
|      8 | computer-networks     |        1 | Computer Networks                            |

<div style="border-left: 4px solid #D97706; background: rgba(217, 119, 6, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#D97706; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">Summed TF is not relevance</strong><br>
Ranking by raw term frequency ignores how rare each term is and how long each document is. It rewards documents that simply repeat query words. Real engines use BM25, which adds inverse document frequency and length normalization. That is exactly what SQLite FTS5 gives us next.
</div>

## SQLite FTS5: A Real In-Database Inverted Index

SQLite ships with FTS5, a full-text search module that stores a genuine inverted
index behind a virtual table and ranks matches with BM25. We index the same
books, then query with the `MATCH` operator and order by the built-in `bm25()`
function.

In [6]:
# A porter-stemming FTS5 virtual table over title and author
con.execute(
    "CREATE VIRTUAL TABLE books USING fts5("
    "  doc_id UNINDEXED, title, author, tokenize='porter unicode61')"
)
for key, record in LIBRARY.items():
    con.execute(
        "INSERT INTO books(doc_id, title, author) VALUES (?, ?, ?)",
        (key, record["title"], record["author"]),
    )
con.commit()
display_md(f"**FTS5 index built** over {len(LIBRARY)} books (title + author).")

**FTS5 index built** over 50 books (title + author).

In [7]:
# bm25() returns a negative score: more negative means a better match,
# so ORDER BY score (ascending) puts the best documents first.
def fts5_search(match_query, limit=10):
    sql = (
        "SELECT doc_id, title, bm25(books) AS score FROM books "
        "WHERE books MATCH ? ORDER BY score LIMIT ?"
    )
    return con.execute(sql, (match_query, limit)).fetchall()

rows = [[i, doc_id, f"{-score:.4f}", title[:46]]
        for i, (doc_id, title, score) in enumerate(fts5_search("computer OR design"), 1)]
display_md(
    "**FTS5 ranked search:** `computer OR design`\n\n"
    "(showing `-bm25()` so that a larger number means a better match)"
)
print_table(rows, headers=["Rank", "Doc ID", "BM25", "Title"])

**FTS5 ranked search:** `computer OR design`

(showing `-bm25()` so that a larger number means a better match)

|   Rank | Doc ID                |   BM25 | Title                                          |
|-------:|:----------------------|-------:|:-----------------------------------------------|
|      1 | computer-organization | 3.4663 | Computer Organization and Design: The Hardware |
|      2 | everyday-things       | 2.9687 | The Design of Everyday Things                  |
|      3 | computer-networks     | 2.1154 | Computer Networks                              |
|      4 | theory-of-computation | 1.9988 | Introduction to the Theory of Computation      |
|      5 | sicp                  | 1.6377 | Structure and Interpretation of Computer Progr |
|      6 | computer-systems      | 1.502  | Computer Systems: A Programmer's Perspective   |

In [8]:
# FTS5 supports the full Boolean and phrase query language
examples = [
    ("computer systems", "implicit AND: both terms required"),
    ("computer OR network", "OR: either term"),
    ("computer NOT network", "NOT: exclude a term"),
    ('"programming languages"', "phrase: adjacent terms"),
    ("comp*", "prefix: any term starting with comp"),
]
rows = []
for q, desc in examples:
    hits = con.execute("SELECT COUNT(*) FROM books WHERE books MATCH ?", (q,)).fetchone()[0]
    rows.append([q, desc, hits])
display_md("**FTS5 query language** (same operators as a dedicated search engine):")
print_table(rows, headers=["MATCH query", "Meaning", "Hits"])

**FTS5 query language** (same operators as a dedicated search engine):

| MATCH query             | Meaning                             |   Hits |
|:------------------------|:------------------------------------|-------:|
| computer systems        | implicit AND: both terms required   |      1 |
| computer OR network     | OR: either term                     |      6 |
| computer NOT network    | NOT: exclude a term                 |      4 |
| "programming languages" | phrase: adjacent terms              |      1 |
| comp*                   | prefix: any term starting with comp |      6 |

### Does FTS5 Agree with Our Hand-Rolled BM25?

FTS5 ranks with BM25, the same model we implemented by hand in ch04-02. The
absolute scores differ (different IDF variant and tokenizer details), but the
rankings should line up closely.

In [9]:
from shared.retrieval import bm25_score, document_frequencies, average_doc_length

# Our BM25 over the same title+author text (stemmed with our pipeline)
corpus = {k: pipeline(f"{r['title']} {r['author']}") for k, r in LIBRARY.items()}
df = document_frequencies(corpus)
adl = average_doc_length(corpus)
N = len(corpus)

q_tokens = pipeline("computer systems design")
ours = sorted(
    ((k, bm25_score(q_tokens, toks, df, N, adl, k=1.2, b=0.75)) for k, toks in corpus.items()),
    key=lambda x: -x[1],
)
our_ranking = [k for k, s in ours if s > 0]

# FTS5 over the same query with OR so both engines see the same candidate set
fts_ranking = [doc_id for doc_id, _, _ in fts5_search("computer OR systems OR design", limit=20)]

rows = []
for rank in range(min(len(our_ranking), len(fts_ranking))):
    ours_id = our_ranking[rank]
    fts_id = fts_ranking[rank]
    rows.append([rank + 1, ours_id, fts_id, "=" if ours_id == fts_id else ""])
display_md("### Ranking comparison: our BM25 vs. SQLite FTS5 BM25")
print_table(rows, headers=["Rank", "Our BM25", "FTS5 BM25", "Match"])
display_md(
    "Small differences are expected: FTS5 and our implementation use slightly "
    "different IDF variants and tokenizers, but both are BM25 and agree on the "
    "top documents."
)

### Ranking comparison: our BM25 vs. SQLite FTS5 BM25

|   Rank | Our BM25              | FTS5 BM25             | Match   |
|-------:|:----------------------|:----------------------|:--------|
|      1 | computer-organization | computer-organization | =       |
|      2 | computer-systems      | computer-systems      | =       |
|      3 | everyday-things       | everyday-things       | =       |
|      4 | theory-of-computation | operating-systems     |         |
|      5 | computer-networks     | computer-networks     | =       |
|      6 | operating-systems     | theory-of-computation |         |
|      7 | database-systems      | database-systems      | =       |
|      8 | sicp                  | sicp                  | =       |

Small differences are expected: FTS5 and our implementation use slightly different IDF variants and tokenizers, but both are BM25 and agree on the top documents.

## PostgreSQL Full-Text Search

Most applications that search inside a database use PostgreSQL rather than
SQLite. The concepts are the same; only the syntax changes. PostgreSQL stores an
indexed document as a `tsvector` (sorted, normalized lexemes) and a query as a
`tsquery` (a Boolean expression of lexemes), matched with the `@@` operator. A
GIN index (Generalized Inverted Index) provides the speed and is a genuine
inverted index: a B-tree of lexemes, each pointing to a posting list of row IDs.

```sql
-- 1. Table with a tsvector column for the search index
CREATE TABLE books (
  id     TEXT PRIMARY KEY,
  title  TEXT NOT NULL,
  author TEXT NOT NULL,
  genre  TEXT,
  year   INT,
  tsv    tsvector
);

-- 2. Build the index. setweight labels title matches (A) above author (B),
--    so ts_rank can weight them differently.
UPDATE books
SET tsv = setweight(to_tsvector('english', title),  'A') ||
          setweight(to_tsvector('english', author), 'B');

CREATE INDEX books_tsv_idx ON books USING GIN(tsv);

-- 3. Ranked search combined with an ordinary SQL predicate on year
SELECT id, title, year,
       ts_rank(tsv, to_tsquery('english', 'computer & systems')) AS r
FROM books
WHERE tsv @@ to_tsquery('english', 'computer & systems')
  AND year < 2010
ORDER BY r DESC
LIMIT 10;
```

Because the `tsvector` and its GIN index live inside the database, full-text
predicates combine naturally with the rest of SQL: a single query can match
text, filter on other columns, join related tables, and run inside a
transaction. That integration is the main reason to search in the database.

<div style="border-left: 4px solid #D97706; background: rgba(217, 119, 6, 0.06); padding: 0.6em 0.9em; margin: 0.6em 0; border-radius: 4px;">
<strong style="color:#D97706; text-transform:uppercase; font-size:0.78em; letter-spacing:0.06em;">PostgreSQL ranking is not BM25</strong><br>
<code>ts_rank</code> scores a match by the frequency of the query lexemes in the document, optionally weighted by the A/B/C/D field labels; <code>ts_rank_cd</code> adds a proximity component. Neither uses a corpus-wide inverse document frequency, so neither is TF-IDF or BM25. Treat in-database search as excellent for term matching and filtering, and only adequate for relevance ranking out of the box. If BM25 matters, use a search extension or move to a Lucene-based engine (next demo).
</div>

## When to Search in the Database

| Criterion | In-database FTS (SQLite / PostgreSQL) | Dedicated engine (Lucene / Elasticsearch) |
|-----------|--------------------------------------|--------------------------------------------|
| Scale | Up to a few million documents | Millions to billions |
| Setup | None: already in your database | Separate service to run and sync |
| Transactions | Full ACID | Eventual consistency |
| Relevance | Matching strong; ranking basic (SQLite has BM25, PostgreSQL does not) | BM25 by default, extensive tuning |
| Combined queries | Text + SQL predicates + joins in one query | Text-first; joins are awkward |
| Best when | Search is a feature of an app you already back with a database | Search is the product |

## Cleanup

In [10]:
con.close()
display_md("**Closed** the in-memory database.")

**Closed** the in-memory database.

---

## Summary

| Concept | Key Point |
|---------|----------|
| Inverted index as SQL | A (term, doc_id, tf) table with a B-tree index on term |
| Boolean in SQL | AND = GROUP BY doc_id HAVING COUNT(DISTINCT term) = k |
| Ranked in SQL | Join postings and sum a per-term weight |
| Summed TF | Rewards repetition; ignores rarity and length (not relevance) |
| SQLite FTS5 | Built-in inverted index with MATCH queries and bm25() ranking |
| bm25() sign | Negative; ORDER BY score ascending puts best matches first |
| PostgreSQL | tsvector + tsquery + @@, GIN index, ts_rank (not BM25) |
| setweight | A/B/C/D field labels that bias ts_rank, like field boosts |
| In-database FTS | Great for matching and combined queries; ranking is basic |

---

## Try It Yourself

1. Add a `genre` column to the FTS5 table and search only within one genre.
   Hint: FTS5 supports column filters like `title:computer`.

2. Change the hand-built SQL ranking to sum a TF-IDF weight instead of raw TF.
   You will need a second query to compute each term's document frequency.

3. Compare `computer systems` (implicit AND) with `computer OR systems` in FTS5.
   How does the size of the result set change, and how does that affect ranking?

In [11]:
# Exercise: FTS5 column filter and OR vs AND result sizes
con2 = sqlite3.connect(":memory:")
con2.execute("CREATE VIRTUAL TABLE b USING fts5(doc_id UNINDEXED, title, tokenize='porter')")
for key, record in LIBRARY.items():
    con2.execute("INSERT INTO b(doc_id, title) VALUES (?, ?)", (key, record["title"]))
con2.commit()

rows = []
for q in ["computer systems", "computer OR systems", "title:design"]:
    hits = con2.execute("SELECT COUNT(*) FROM b WHERE b MATCH ?", (q,)).fetchone()[0]
    rows.append([q, hits])
display_md("**Result-set sizes** for different FTS5 queries:")
print_table(rows, headers=["MATCH query", "Hits"])
con2.close()

**Result-set sizes** for different FTS5 queries:

| MATCH query         |   Hits |
|:--------------------|-------:|
| computer systems    |      1 |
| computer OR systems |      7 |
| title:design        |      2 |